<a href="https://colab.research.google.com/github/Ekimxii/fitness_pose_tracker/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workout coach app (Gradio)

Upload a workout video and pick the exercise. The app returns the video with the skeleton drawn on, the rep count, and the coach's messages.

**Run the cells top to bottom in Colab.** The last cell prints a public link to the app.

## 1. Setup
Gets the latest repo (code, models, data) and installs MediaPipe. Gradio and the Claude library already come with Colab.
If `import gradio` fails later with a pillow error, run `!pip install -q "pillow<13"`, then Runtime > Restart session.

In [ ]:
import os, sys

REPO = "/content/fitness_pose_tracker"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Ekimxii/fitness_pose_tracker.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
sys.path.insert(0, REPO)   # so `import feedback` finds the repo's feedback.py

!pip install -q mediapipe

## 2. Claude API key
Add your key as a Colab secret named `client` (key icon in the left sidebar, then turn on notebook access). Never paste the key into the notebook.

In [ ]:
from google.colab import userdata
import anthropic

client = anthropic.Anthropic(api_key=userdata.get("client"))

## 3. Feedback code
Loads the gate, retrieval and message writing from `feedback.py`, plus the 2,013 real coach messages and their embeddings.

In [ ]:
from feedback import load_corpus, FeedbackController, retrieve_top3, make_message

feedback_df, feedback_embeddings, encoder = load_corpus()

## 4. Form pipeline
Runs `live.py` on the video without a window. It does pose detection, rep counting and the form checks, and saves an overlay video, the issue dicts (`_form.json`) and the rep table (`_reps.csv`).
Each run gets its own folder, so files from earlier runs never get mixed in.

In [ ]:
import glob, json, subprocess, time
import pandas as pd

def ffmpeg_h264(src, dst):
    """Re-encode a video as H.264 MP4, which both OpenCV and browsers can read."""
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-an",
                    "-c:v", "libx264", "-preset", "veryfast", "-pix_fmt", "yuv420p", dst], check=True)
    return dst

def newest(folder, pattern):
    """The newest file in `folder` matching `pattern`, or None."""
    found = glob.glob(os.path.join(folder, pattern))
    return max(found, key=os.path.getmtime) if found else None

def run_form_pipeline(video_path, exercise):
    """Run live.py on one video. Returns (overlay video path, issue dicts, reps counted)."""
    out_dir = f"Video Output/app/{int(time.time())}"
    os.makedirs(out_dir, exist_ok=True)
    clip = ffmpeg_h264(video_path, os.path.join(out_dir, "input.mp4"))   # also handles .mov and webcam recordings
    run = subprocess.run([sys.executable, "live.py", "--video", clip, "--no-window",
                          "--exercise", exercise, "--output", out_dir], capture_output=True, text=True)
    if run.returncode != 0:
        raise RuntimeError("live.py failed:\n" + run.stderr[-2000:])

    form, reps = newest(out_dir, "*_form.json"), newest(out_dir, "*_reps.csv")
    dicts = json.load(open(form)) if form else []
    rep_count = len(pd.read_csv(reps)) if reps else 0
    return newest(out_dir, "*_overlay.mp4"), dicts, rep_count

## 5. Coach messages
For each issue dict: the gate decides whether to speak, retrieval finds the 3 closest real coach messages, and `make_message` has Claude write the message (a correction, praise for fixing an issue, or encouragement) in the style of those real messages.

In [ ]:
def coach_messages(dicts):
    """One row per coach message: time, rep, type, issue, message."""
    controller = FeedbackController()
    rows = []
    for d in dicts:
        decision = controller.process(d)
        if decision is None:
            continue   # the gate decided to stay quiet
        top3 = retrieve_top3(decision, feedback_df, feedback_embeddings, encoder)
        message = make_message(decision, top3, client)
        if message:
            rows.append({"time (s)": round(d["t"], 1), "rep": d["rep"], "type": decision["action"],
                         "issue": decision.get("issue", ""), "message": message})
    return pd.DataFrame(rows, columns=["time (s)", "rep", "type", "issue", "message"])

## 6. Put it together and test on one clip
`coach` is the one function the app calls. Test it first: upload a clip with the folder icon in the left sidebar, then set `TEST_CLIP` to its path.

In [ ]:
import gradio as gr

def coach(video_path, exercise):
    """Video + exercise in; playable overlay video, rep count and coach messages out."""
    if not video_path:
        raise gr.Error("Upload or record a video first.")
    overlay, dicts, rep_count = run_form_pipeline(video_path, exercise)
    if overlay is None:
        raise gr.Error("No video came out. Check that your whole body is in frame.")
    playable = ffmpeg_h264(overlay, overlay.replace(".mp4", "_h264.mp4"))
    return playable, rep_count, coach_messages(dicts)

TEST_CLIP = "/content/my_squats.mp4"   # change to your clip's path
if os.path.exists(TEST_CLIP):
    video, rep_count, messages = coach(TEST_CLIP, "squats")
    display(pd.DataFrame({"reps counted": [rep_count], "messages": [len(messages)]}))
    display(messages)
else:
    display(pd.DataFrame({"skipped": [f"No file at {TEST_CLIP}. Upload a clip to test, or go straight to the app."]}))

## 7. Launch the app
Prints a public link that lasts 72 hours. Anyone with the link can use the app, and every correction uses the API key's credits, so only share it with the team.

In [ ]:
EXERCISES = ["squats", "pushups", "situp", "jumping_jacks", "pullups"]

demo = gr.Interface(
    fn=coach,
    inputs=[gr.Video(label="Workout video (upload or record)"),
            gr.Dropdown(EXERCISES, value="squats", label="Exercise")],
    outputs=[gr.Video(label="Your reps"),
             gr.Number(label="Reps counted"),
             gr.Dataframe(label="Coach messages")],
    title="Workout coach",
    description="Use a level camera with your whole body in frame, pick the exercise, and press Submit.",
)
demo.launch(share=True, debug=True)

## 8. Before the presentation
- Run 1–2 real clips through the app.
- Screen-record it as a backup (Cmd + Shift + 5 on a Mac), in case Colab or the API is slow during the demo.